#### MetaQuest3 wrist attach + collision-aware arm IK -> `FFW SG2`


In [7]:
%run ../package/init_project.py

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *

app = get_qt_app()



[ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /Users/jaehoon/VAI_test/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation


In [8]:
native_backend = build_sg2_collision_aware_ik_backend(
    clean   = False,
    verbose = True,
    required = True,)
print("bind info:",native_backend.native_info())


[build_native]
  module      : _mujoco_collision_aware_ik_bind
  output      : mujoco_sim/so/_mujoco_collision_aware_ik_bind.cpython-310-darwin.so
  compiler    : clang++
  project cpp:
    [01] mujoco_sim/cpp/bindings.cpp
    [02] mujoco_sim/cpp/signed_distance_bindings.cpp
    [03] mujoco_sim/cpp/signed_distance.cpp
    [04] mujoco_sim/cpp/ik_bindings.cpp
    [05] mujoco_sim/cpp/ik.cpp
  project hpp:
    [01] mujoco_sim/cpp/signed_distance_bindings.hpp
    [02] mujoco_sim/cpp/signed_distance.hpp
    [03] mujoco_sim/cpp/ik_bindings.hpp
    [04] mujoco_sim/cpp/ik.hpp
    [05] mujoco_sim/cpp/mujoco_handles.hpp
  defines:
    [01] RI_NATIVE_ENABLE_SIGNED_DISTANCE
    [02] RI_NATIVE_ENABLE_IK
    [03] RI_NATIVE_MODULE_NAME=_mujoco_collision_aware_ik_bind
  nanobind cpp:
    [01] nanobind/src/nb_combined.cpp
  commands    : hidden; pass show_commands=True to print full commands
[build_native] compile
  [01/06] mujoco_sim/cpp/bindings.cpp -> build/native/bindings.o
  [02/06] mujoco_sim/cpp/

In [ ]:
#ip_address_metaquest3 = "192.168.6.100"
ip_address_metaquest3 = "192.168.0.45"
metaquest3_port = 5555

socket_sub = create_zmq_sub_socket(
    ip_address                 = ip_address_metaquest3,
    port                       = metaquest3_port,
    wait_seconds_after_connect = 0,)
print(f"MetaQuest3: tcp://{ip_address_metaquest3}:{metaquest3_port}")


MetaQuest3: tcp://192.168.6.100:5555


In [10]:
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
    ],
    output_xml_path = str(SG2_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)


[merge_mjcfs] Merging [2] MJCF files:
  - [0] [/Users/jaehoon/VAI_test/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/floor_simple_white.xml]
  - [1] [/Users/jaehoon/VAI_test/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2.xml]
[merge_mjcfs] Saved merged XML to:[/Users/jaehoon/VAI_test/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml]
MuJoCo model loaded from [/Users/jaehoon/VAI_test/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].


In [11]:
input_hz = 20
render_hz = 20
max_missing_quest_ticks = quest_missing_tick_limit(
    input_hz              = input_hz,
    missing_grace_seconds = 1.0,)

hand_types = list(SG2_HAND_TYPES)
hand_labels = dict(SG2_HAND_LABELS)

# Reset
close_objects("rp")
env.reset()
env.set_p("base_link","body",(0.0,0.0,0.01))
env.forward(
    q           = [np.deg2rad(-90.0),np.deg2rad(-90.0)],
    joint_names = ["arm_l_joint4","arm_r_joint4"],)

command_joint_names = filter_existing_joint_names(env,SG2_REAL_COMMAND_JOINT_NAMES)
q_use = env.get_qpos(joint_names=command_joint_names)
q_use_min = env.get_joint_mins(command_joint_names)
q_use_max = env.get_joint_maxs(command_joint_names)

collision_geom_pairs,collision_candidate_info = create_sg2_collision_candidate_geom_pairs(
    env               = env,
    init_exclude_dist = 0.05,)
print_sg2_collision_candidate_summary(collision_candidate_info)

rp = MuJoCoRenderProcess(
    0.22,0.00,1.0,1.0,
    xml_path    = xml_path,
    title       = "MetaQuest3 wrist IK -> FFW SG2",
    render_hz   = render_hz,
    max_sec     = 300.0,
    profile     = False,
    transparent = True,)
rp.start()

collision_ik_solvers,arm_joint_names = create_sg2_collision_aware_arm_ik_solvers(
    env                = env,
    mujoco_parser_cls  = MuJoCoParser,
    body_ik_solver_cls = BodyPositionIKSolverRevPriBase,
    enable_depenetration = False,)
active_ik_solver = collision_ik_solvers["tracking"]
collision_ik_result = make_idle_collision_ik_result(active_ik_solver,safe_margin=0.05)

T_quest_start = env.get_T(name="head_link2",type="body")
quest_retargeting = QuestPalmRetargeting(start_T=T_quest_start)

T_quest_pose_by_idx = {}
T_virtual_quest_pose_by_idx = {}
T_quest_wrist = {hand_type:None for hand_type in hand_types}
T_raw_wrist_target = {hand_type:None for hand_type in hand_types}
T_attach_wrist_target = {hand_type:None for hand_type in hand_types}
T_virtual_quest_wrist = {hand_type:None for hand_type in hand_types}

quest_missing_ticks = {hand_type:max_missing_quest_ticks+1 for hand_type in hand_types}
quest_all_missing_ticks = max_missing_quest_ticks+1
quest_attach_state = {
    hand_type:create_quest_wrist_attach_state(
        hand_type             = hand_type,
        attach_seconds        = 2.5,
        attach_start_distance = 0.20,)
    for hand_type in hand_types
}
last_solved_wrist_target = {hand_type:None for hand_type in hand_types}
arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
wrist_target_motion = {hand_type:(np.inf,np.inf) for hand_type in hand_types}
gripper_q_by_hand = {hand_type:0.0 for hand_type in hand_types}
gripper_info_by_hand = {hand_type:{} for hand_type in hand_types}
detach_gesture_state = create_sg2_detach_gesture_state(
    hand_types   = hand_types,
    finger_names = ("middle","ring","little"),)
for hand_type in hand_types:
    set_sg2_gripper_qpos(env,hand_type,gripper_q_by_hand[hand_type])

# Loop
tmr_update = SimpleTimer(name="Update",Hz=input_hz,verbose=True)
tmr_render = SimpleTimer(name="Render",Hz=render_hz,verbose=True)
tmr_update.start()
tmr_render.start()
env.reset_wall_time()

for hand_type in hand_types:
    print("%s arm IK joints:"%hand_type,len(arm_joint_names[hand_type]),arm_joint_names[hand_type])
print("collision IK joints:",active_ik_solver.joints_use)
print("Depenetration IK: disabled.")
print("Quest thumb-index pinch -> SG2 gripper: enabled.")
print("Detach gesture: bend middle/ring/little on both hands.")


Collision candidates
--------------------
robot collision geom:[34]
collision geom:[35]
candidate geom pairs:[493]
excluded initial geom pairs:[37]
render process pid:[46802] port:[49565] target_hz:[20.0]
render process log:[/var/folders/x1/6b_c7v154pnfsmpfg7hh11mm0000gn/T/ri_motion_render_7ntiq8ce.log]
MuJoCo model loaded from [/Users/jaehoon/VAI_test/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].
[Update] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[Render] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[Update] Start ([20.0]HZ. MAX_SEC:[inf]sec).
[Render] Start ([20.0]HZ. MAX_SEC:[inf]sec).
left arm IK joints: 7 ['arm_l_joint1', 'arm_l_joint2', 'arm_l_joint3', 'arm_l_joint4', 'arm_l_joint5', 'arm_l_joint6', 'arm_l_joint7']
right arm IK joints: 7 ['arm_r_joint1', 'arm_r_joint2', 'arm_r_joint3', 'arm_r_joint4', 'arm_r_joint5', 'arm_r_joint6', 'arm_r_joint7']
collision IK joints: ['arm_l_joint1', 'arm_l_joint2', 'arm_l_joint3', 'arm_l_joint4', 'arm_l

In [12]:
try:
    while rp.is_running():
        update_qt()
        env.increase_wall_time()

        if tmr_update.do_run():
            # Receive MetaQuest3
            T_quest_wrist,T_quest_pose_by_idx,quest_missing_ticks,quest_all_missing_ticks,quest_updated,_ = update_metaquest3_wrist_data(
                socket_sub              = socket_sub,
                quest_retargeting       = quest_retargeting,
                hand_types              = hand_types,
                T_quest_wrist           = T_quest_wrist,
                T_quest_pose_by_idx     = T_quest_pose_by_idx,
                quest_missing_ticks     = quest_missing_ticks,
                quest_all_missing_ticks = quest_all_missing_ticks,
                quest_attach_state      = quest_attach_state,
                check_distance          = 0.10,)

            # Attach Quest wrist to SG2 wrist
            arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
            (
                T_raw_wrist_target,
                T_attach_wrist_target,
                T_virtual_quest_wrist,
                attached_wrist_targets,
                quest_attach_state,
                last_solved_wrist_target,
                wrist_target_motion,
            ) = update_sg2_wrist_attach_targets(
                env                      = env,
                hand_types               = hand_types,
                T_quest_wrist            = T_quest_wrist,
                quest_missing_ticks      = quest_missing_ticks,
                quest_updated            = quest_updated,
                quest_attach_state       = quest_attach_state,
                last_solved_wrist_target = last_solved_wrist_target,
                max_missing_quest_ticks  = max_missing_quest_ticks,
                dt                       = getattr(tmr_update,"dt",1.0/input_hz),
                attach_seconds           = 3.0,
                attach_start_distance    = 0.20,)

            # Quest thumb-index pinch -> SG2 gripper
            gripper_q_by_hand,gripper_info_by_hand = update_sg2_gripper_from_quest_pinch(
                env                     = env,
                T_pose_by_idx           = T_quest_pose_by_idx,
                hand_types              = hand_types,
                quest_missing_ticks     = quest_missing_ticks,
                max_missing_quest_ticks = max_missing_quest_ticks,
                prev_gripper_q_by_hand  = gripper_q_by_hand,
                pinch_closed_distance   = 0.025,
                pinch_open_distance     = 0.095,
                smoothing_alpha         = 0.35,)


            # Middle/ring/little bend on both hands -> detach
            detach_gesture_state = update_sg2_detach_gesture_state(
                T_pose_by_idx           = T_quest_pose_by_idx,
                hand_types              = hand_types,
                quest_missing_ticks     = quest_missing_ticks,
                max_missing_quest_ticks = max_missing_quest_ticks,
                dt                      = getattr(tmr_update,"dt",1.0/input_hz),
                detach_gesture_state    = detach_gesture_state,
                finger_names            = ("middle","ring","little"),
                enter_bend_ratio        = 0.45,
                exit_bend_ratio         = 0.28,
                mean_enter_bend_ratio   = 0.40,
                mean_exit_bend_ratio    = 0.25,
                min_closed_finger_count = 2,
                hold_seconds_to_detach  = 1.2,)
            if detach_gesture_state["should_detach"]:
                quest_attach_state = apply_sg2_detach_gesture_to_attach_state(
                    quest_attach_state = quest_attach_state,
                    hand_types         = hand_types,)
                attached_wrist_targets = {}
                T_attach_wrist_target = {hand_type:None for hand_type in hand_types}
                T_virtual_quest_wrist = {hand_type:None for hand_type in hand_types}
                last_solved_wrist_target = {hand_type:None for hand_type in hand_types}

            # Collision-aware arm IK
            collision_ik_result = solve_sg2_collision_aware_wrist_ik(
                env                    = env,
                tracking_solver        = collision_ik_solvers["tracking"],
                depenetration_solver   = None,
                wrist_targets_by_hand  = attached_wrist_targets,
                collision_geom_pairs   = collision_geom_pairs,
                native_backend         = native_backend,
                safe_margin            = 0.05,
                signed_distance_topk   = 5,
                enable_depenetration   = False,)
            active_ik_solver = collision_ik_result["solver"]
            last_solved_wrist_target,arm_ik_info_by_hand = update_last_solved_wrist_targets(
                last_solved_wrist_target = last_solved_wrist_target,
                arm_ik_info_by_hand      = arm_ik_info_by_hand,
                attached_wrist_targets   = attached_wrist_targets,
                ik_info                  = collision_ik_result["ik_info"],)

            q_use = np.clip(env.get_qpos(joint_names=command_joint_names),q_use_min,q_use_max)
            env.forward(q=q_use,joint_names=command_joint_names)

            T_virtual_quest_pose_by_idx = {}
            if quest_all_missing_ticks <= max_missing_quest_ticks:
                T_virtual_quest_pose_by_idx = virtualize_quest_pose_dicts_from_wrists(
                    T_pose_by_idx = T_quest_pose_by_idx,
                    T_quest_wrist = T_quest_wrist,
                    T_quest_target = T_attach_wrist_target,
                    hand_types     = hand_types,)
            tmr_update.end()

        if tmr_render.do_run():
            draw_cmds = make_sg2_metaquest_draw_cmds(
                rp                    = rp,
                env                   = env,
                hand_types            = hand_types,
                hand_labels           = hand_labels,
                quest_missing_ticks   = quest_missing_ticks,
                T_quest_wrist         = T_quest_wrist,
                T_virtual_quest_wrist = T_virtual_quest_wrist,
                collision_ik_result   = collision_ik_result,
                active_ik_solver      = active_ik_solver,
                plot_signed_distance  = False,
                plot_ik_targets       = False,)
            draw_cmds.extend(make_sg2_gripper_draw_cmds(rp,gripper_info_by_hand))
            overlay_cmds = [
                rp.overlay("Attach",make_sg2_attach_overlay_text(hand_types,hand_labels,quest_attach_state),loc="top left"),
                rp.overlay("Gripper",make_sg2_gripper_overlay_text(hand_types,hand_labels,gripper_info_by_hand),loc="top left"),
                rp.overlay("Gesture",make_sg2_detach_gesture_overlay_text(hand_types,hand_labels,detach_gesture_state),loc="top left"),
                rp.overlay("IK",make_sg2_ik_overlay_text(collision_ik_result),loc="top left"),
                rp.overlay("Collision",make_sg2_collision_overlay_text(collision_ik_result),loc="top left"),
            ]
            rp.render(env=env,overlays=overlay_cmds,draws=draw_cmds,mode="snapshot",profile=False)
            tmr_render.end()
            sleep(0.001)

except KeyboardInterrupt:
    print("Interrupted.")
finally:
    close_zmq_socket(socket_sub)
    close_objects("rp")


[Update][17][0.8s] delayed! T:[50.0]ms interval:[57.1]ms prev_run:[0.9]ms.
[Render][17][0.8s] delayed! T:[50.0]ms interval:[58.5]ms prev_run:[0.9]ms.


In [13]:
if getattr(env,"final_rgb_img",None) is not None:
    imshow(env.final_rgb_img,fig_w=6,title="Final scene",title_fs=8)
print("Done.")


Done.
